# Baomai: confirm the annotation result with seed 46

**Select a GPU runtime, run all cells, then return one ZIP.** This notebook performs the single predeclared seed 46 confirmation of the same reviewed annotations used for seed45. It runs a short resource trial and one fresh control/candidate pair. No seed 46 result is precomputed.

Both arms start from the original **epoch173** checkpoint using `reviewed-annotations-seed46-v1`: **seed 46, 30 epochs, batch 4, image size 1280, FP32, AdamW, nominal learning rate 0.0001 and bias warmup learning rate 0.0001**. The final epoch-30 `last.pt` is selected in advance. The numerical recipe differs from seed 45 only in its seed.

| Run | Training labels | Initialization |
| --- | --- | --- |
| Control (`reference`) | Original frozen labels | Original epoch173 |
| Candidate | Unchanged `annotations-2026-11-10` overlay | Original epoch173 |

The candidate still contains five added fronts, six deleted rows and three retained-row modifications across eight images. All original visible corner marks survive; two added marks remain the only new visible endpoints. Neither a seed 45 model nor the resource trial initializes these runs. Original validation labels remain identical in both arms.

Training implementation revision: `23d7db97ca626f5591dbd26e699c0f23d318ee73`. The first cell refuses an unpinned draft before any GPU or filesystem action. The [seed 46 handoff](../docs/training-annotations-seed46.md) describes the fixed procedure. Candidate manifest SHA-256 remains `28fc8f5da95605089d61fb062132200ed296c3a10b216f1a61dbda1798ea4ba7`.

Seed 45 and seed 46 are reported separately on original and PR30 revised references. Preserve both favorable and unfavorable results. This notebook neither searches additional seeds nor automatically promotes a model.

Finished trial and arm directories are copied to Google Drive at `MyDrive/baomai-reviewed-annotations-seed46-20261111` after each successful `train_one`, and the results ZIP is copied there after packaging. Approve the Drive mount when asked. If the runtime dies, start a new GPU session and Run all again: finished folders already on Drive are restored into the new workspace and are not trained again.


## 1. Select a GPU and create a fresh workspace

In Colab, select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. Keep the session unchanged until both runs and packaging finish. Approve the Google Drive mount in this cell so finished runs survive an ephemeral `/content` disk.

Prefer the same **NVIDIA L4** environment as seed 45 when available: Python 3.13.16, Torch 2.14.1+cu130, torchvision 0.29.1+cu130 and CUDA 13.0. The requirement is a matching supported CUDA runtime for the two arms; there is no A100 requirement. If the provider supplies a different environment, retain its exact recorded identity and interpret that cross-seed difference explicitly.

The notebook creates `/content/baomai-reviewed-annotations-seed46`. It still refuses an existing `/content` workspace in the same session. After an interrupted run that left `/content` intact, retry that arm in a new folder. If the session dies and `/content` is gone, start a new GPU runtime and Run all: finished runs on Drive are copied back into the new `RUN_BASE`, and those runs are not trained again.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

SOURCE_REVISION = "23d7db97ca626f5591dbd26e699c0f23d318ee73"
RECIPE_ID = "reviewed-annotations-seed46-v1"
RECIPE_SHA256 = "f18e0a813a1057e1b3ab3a44ea3ce9b2a3e7673cc2bf04ca09b394bfdd8c51b8"
CANDIDATE_MANIFEST = "data/candidates/annotations-2026-11-10/manifest.json"
CANDIDATE_SHA256 = "28fc8f5da95605089d61fb062132200ed296c3a10b216f1a61dbda1798ea4ba7"
RUN_BASE = Path("/content/baomai-reviewed-annotations-seed46")

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION), "Use the handoff notebook with its full source commit pinned."
assert sys.version_info >= (3, 11), "Python 3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

run([sys.executable, "-c", "import torch, torchvision; "
     "assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before continuing.'; "
     "print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda); "
     "print('GPU:', torch.cuda.get_device_name(0)); "
     "print('CUDA calculation:', (torch.ones(1, device='cuda:0') * 2).item())"])
run(["nvidia-smi"])
DRIVE_ROOT = None
try:
    from google.colab import drive
except ImportError:
    print("Google Drive mount is unavailable; completed runs stay under the local workspace only.")
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/baomai-reviewed-annotations-seed46-20261111")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive folder:", DRIVE_ROOT)

def save_to_drive(path):
    if DRIVE_ROOT is None:
        return
    path = Path(path)
    destination = DRIVE_ROOT / path.relative_to(RUN_BASE)
    marker = Path(str(destination) + ".complete")
    if destination.exists() and marker.is_file():
        print("Drive already has", destination, flush=True)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_dir():
        shutil.rmtree(destination)
    elif destination.exists():
        destination.unlink()
    marker.unlink(missing_ok=True)
    print("Copying to Drive:", destination, flush=True)
    if path.is_dir():
        shutil.copytree(path, destination, copy_function=shutil.copyfile)
    else:
        shutil.copyfile(path, destination)
    marker.write_text(path.relative_to(RUN_BASE).as_posix() + "\n", encoding="utf-8")
    print("Saved to Drive:", destination, flush=True)

def restore_from_drive(folder):
    if DRIVE_ROOT is None:
        return None
    source = DRIVE_ROOT / folder
    if not source.is_dir() or not Path(str(source) + ".complete").is_file():
        return None
    destination = RUN_BASE / folder
    if not destination.exists() and not destination.is_symlink():
        print("Restoring from Drive:", source, flush=True)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination, copy_function=shutil.copyfile)
    return destination

if RUN_BASE.exists() or RUN_BASE.is_symlink():
    raise FileExistsError(f"Workspace already exists: {RUN_BASE}. Preserve its results and choose a new RUN_BASE.")
RUN_BASE.mkdir(parents=True, exist_ok=False)
REPO = RUN_BASE / "dl"
print("Workspace:", RUN_BASE)


## 2. Download and verify the exact inputs

Git supplies the pinned code and labels. Selective Git LFS downloads only the **157 original training/validation PNGs and epoch173 weights**. Their inventory, hashes and dimensions are checked before training.

The preflight verifies the exact candidate manifest, row edits, corner transfers, original labels and unchanged validation inputs. You do not need to upload an earlier input ZIP.


In [ ]:
lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
run(["git", "init", REPO], env=git_env)
run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], cwd=REPO)
run(["git", "fetch", "--depth=1", "origin", SOURCE_REVISION], cwd=REPO, env=git_env)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO, env=git_env)
run(["git", "lfs", "install", "--local"], cwd=REPO)
reference_manifest = json.loads((REPO / "data/manifest.json").read_text())
expected_images = {"data/" + row["image"] for row in reference_manifest["records"]}
tracked_images = set(subprocess.check_output(
    ["git", "ls-files", "data/images/train/*.png", "data/images/val/*.png"],
    cwd=REPO, text=True).splitlines())
assert len(reference_manifest["records"]) == len(expected_images) == 157
assert tracked_images == expected_images, "The LFS image selection differs from the frozen manifest."
assert reference_manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
assert reference_manifest["model_epoch"] == 173
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt",
     "--exclude="], cwd=REPO)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_revision == SOURCE_REVISION
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()

input_preflight = """
import json
from pathlib import Path
import sys
sys.dont_write_bytecode = True
root = Path.cwd()
sys.path.insert(0, str(root / 'scripts'))
from evaluate_pose import file_sha256, preflight_dataset
from stage_pose_training import source_state
from pose_checkpoint import get_recipe, recipe_identity
candidate = root / sys.argv[1]
recipe_id, expected_hash, expected_recipe_hash = sys.argv[2:5]
assert file_sha256(candidate) == expected_hash, 'Candidate manifest differs from the pinned handoff.'
proposal = json.loads(candidate.read_text())
assert proposal['kind'] == 'training_annotation_overlay'
assert proposal['candidate_id'] == 'annotations-2026-11-10'
edits = {key: sum(len(change[key]) for change in proposal['changes'])
         for key in ('deletions', 'replacements', 'additions')}
assert len(proposal['changes']) == 8
assert edits == {'deletions': 6, 'replacements': 3, 'additions': 5}, edits
state = source_state(root, candidate)
initial = preflight_dataset(root, require_weights=True)
assert len(state['records']) == 157
assert state['counts']['reference']['train'] == {
    'images': 137, 'objects': 3772, 'visible_endpoints': 6839, 'missing_endpoints': 705, 'empty_images': 13}
assert state['counts']['candidate']['train'] == {
    'images': 137, 'objects': 3771, 'visible_endpoints': 6841, 'missing_endpoints': 701, 'empty_images': 13}
assert state['counts']['reference']['val'] == state['counts']['candidate']['val'] == {
    'images': 20, 'objects': 664, 'visible_endpoints': 1227, 'missing_endpoints': 101, 'empty_images': 3}
assert state['source']['initial_checkpoint_sha256'] == initial['model_sha256']
assert Path(initial['weights']) == root / 'weights/cassette_lower_epoch173.pt'
expected_recipe = get_recipe('warmup-bias-lr-1e-4-v1')
expected_recipe['seed'] = 46
assert get_recipe(recipe_id) == expected_recipe
assert recipe_identity(recipe_id)['recipe_sha256'] == expected_recipe_hash
print(json.dumps({'status': 'verified', 'original_images': len(state['records']),
                  'annotation_edits': edits, 'label_counts': state['counts'],
                  'candidate_manifest_sha256': expected_hash,
                  'initial_checkpoint_sha256': initial['model_sha256'],
                  'weights_verified': initial['weights_verified'],
                  **recipe_identity(recipe_id)}, indent=2))
"""
run([sys.executable, "-c", input_preflight, CANDIDATE_MANIFEST, RECIPE_ID, CANDIDATE_SHA256, RECIPE_SHA256], cwd=REPO)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified clean source:", actual_revision)


## 3. Preserve CUDA PyTorch and install the pinned trainer

The installed `torch` and `torchvision` versions are constrained during installation. If this cell reports a dependency conflict, resolve it before training; do not discard the constraints and silently change the GPU runtime. Training uses fresh subprocesses, so it sees the installed packages without importing the trainer into this notebook's kernel.

In [ ]:
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text(
    ''.join(f'{name}=={version(name)}\n' for name in ('torch', 'torchvision')), encoding='utf-8'
)
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; assert torch.cuda.is_available(); "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__, torch.cuda.get_device_name(0))"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
assert HOST_RSS_LIMIT_MIB >= 6000, "Free more host RAM or select a larger-memory runtime, then rerun this cell."
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Time budgets: 600 seconds for the trial; 21600 seconds per complete run.")


## 4. Run a short real training trial

The trial exercises the candidate labels with the same full 30-epoch schedule used by both later runs, then stops after **four batches**. It must report actual optimizer updates and an observed parameter change before the notebook continues.

Its time budget is 600 seconds. Each complete run has a six-hour stop budget; these limits are not duration estimates. The printed memory limit covers **host RAM**, while GPU VRAM is shown by `nvidia-smi`. This trial does not qualify the later validation or checkpoint-finalization memory costs and does not produce an eligible model.

Progress appears below every 20 seconds; detailed trainer output is retained in each run's `train.log`.

In [ ]:
def train_one(folder, arm, *, trial=False):
    assert arm in ("reference", "candidate")
    output = RUN_BASE / folder

    def assert_finished():
        receipt = json.loads((output / "training-receipt.json").read_text())
        assert receipt["status"] == ("trial-complete" if trial else "complete")
        assert receipt["optimizer_steps"] > 0 and receipt["optimizer_parameter_change_observed"] is True
        assert receipt["arm"] == arm
        assert receipt["recipe_id"] == RECIPE_ID
        assert receipt["initial_checkpoint_sha256"] == reference_manifest["model_sha256"]
        assert receipt["training_settings"]["epochs"] == 30
        assert receipt["training_settings"]["seed"] == 46
        assert receipt["training_settings"]["warmup_bias_lr"] == 0.0001
        if not trial:
            assert receipt["completed_epochs"] == list(range(1, 31))
        assert (output / "checkpoint-manifest.json").exists() is (not trial)

    restore_from_drive(folder)
    if output.exists() or output.is_symlink():
        try:
            assert_finished()
        except Exception as error:
            raise FileExistsError(
                f"Preserve {output}; retry only the failed cell with a new folder name. {error}"
            ) from error
        print("Reusing completed run:", output, flush=True)
        save_to_drive(output)
        return output
    command = [sys.executable, "scripts/train_pose_comparison.py",
               "--trial" if trial else "--train", "--arm", arm, "--recipe", RECIPE_ID,
               "--candidate", CANDIDATE_MANIFEST,
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting:", folder, arm, RECIPE_ID, flush=True)
    print("Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}

    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass

    started = time.monotonic()
    last_notice = -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}: {elapsed:.0f}s; status={progress.get('status')}; "
                          f"completed epochs={len(progress.get('completed_epochs', []))}/30; "
                          f"batches={progress.get('batches_completed', 0)}; "
                          f"updates={progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}: {elapsed:.0f}s; verifying/staging inputs", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            receipt_file = output / "training-receipt.json"
            if receipt_file.exists():
                failed_receipt = json.loads(receipt_file.read_text())
                for key in ("error", "input_verification_error", "checkpoint_manifest_error"):
                    if failed_receipt.get(key):
                        print(f"{key}: {failed_receipt[key]}", flush=True)
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed. Preserve its receipt/log and use a new output directory for a retry.")
    except BaseException:
        # Stop this invocation's descendants if the notebook cell is interrupted.
        # Partial runs remain ineligible; no receipt is fabricated here.
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    assert_finished()
    save_to_drive(output)
    return output

TRIAL = train_one("resource-trial", "candidate", trial=True)
print("Resource trial passed. This is not a completed model.")


## 5. Train the fresh seed 46 control and candidate sequentially

The original-label control runs first, followed by the unchanged annotation candidate. Both use the same seed 46 recipe and start independently from original epoch173. No prior model or resource-trial checkpoint is used as a warm start.

A failed or interrupted run is ineligible for packaging. Preserve its receipt/log and retry in a fresh output directory. If only the candidate needs retrying in the unchanged session, use `CANDIDATE = train_one("candidate-retry-001", "candidate")`, then continue to packaging. If source, runtime or scientific settings change, start a fresh pair. Do not substitute a seed 45 arm for either seed 46 arm.

Each successful `train_one` copies that finished folder to `MyDrive/baomai-reviewed-annotations-seed46-20261111` under the same relative name, including a retry name. A finished Drive copy is not overwritten. If the copy fails after a successful local run, rerun that cell in the same session; it reuses the finished folder and retries the copy. An incomplete local folder still raises `FileExistsError`; retry it with a new folder name.

If the runtime dies and `/content` is gone, start a new GPU session and Run all. Each finished folder this notebook still names is restored into the new `RUN_BASE` and is not trained again. A retry folder is restored only when its cell still uses that name. Do not delete a finished Drive copy to force a retrain. A restored arm stays as it is. Continue the missing arms in a session whose source and runtime match those receipts; otherwise keep the Drive copies and start a fresh pair.


In [ ]:
REFERENCE = train_one("reference", "reference")
CANDIDATE = train_one("candidate", "candidate")
for arm, folder in (("reference", REFERENCE), ("candidate", CANDIDATE)):
    receipt = json.loads((folder / "training-receipt.json").read_text())
    print(arm, "recipe:", receipt["recipe_id"], "epochs:", len(receipt["completed_epochs"]),
          "batches:", receipt["batches_completed"], "optimizer updates:", receipt["optimizer_steps"],
          "GPU:", receipt["runtime"]["device_name"])


## 6. Verify and package the matched results

The existing paired-results packer checks complete-run eligibility, source/runtime identity, the unchanged candidate and shared numerical settings. It includes each finalized epoch-30 `last.pt`, receipt, checkpoint manifest, staging manifest, CSV results and raw log. Failed, incomplete and mixed-seed pairs are refused.

The standard return ZIP contains the two full-run bundles. Keep the separate resource-trial receipt/log in the workspace; it is not included in this ZIP. Packaging records the experiment and does not approve a model. Original images and intermediate checkpoints need not be returned.

An existing ZIP is never overwritten. After the ZIP is written, it is copied to `MyDrive/baomai-reviewed-annotations-seed46-20261111`; an existing finished Drive copy is left in place.


In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-reviewed-annotations-seed46-results.zip"
run([sys.executable, "scripts/package_gpu_results.py", "--reference", REFERENCE,
     "--candidate", CANDIDATE, "--output", RESULTS_ZIP], cwd=REPO)
with RESULTS_ZIP.open("rb") as stream:
    digest = hashlib.file_digest(stream, "sha256").hexdigest()
print("Return:", RESULTS_ZIP.name)
print("Bytes:", RESULTS_ZIP.stat().st_size)
print("SHA-256:", digest)
save_to_drive(RESULTS_ZIP)


## 7. Download and return the ZIP

Attach **`baomai-reviewed-annotations-seed46-results.zip`** in chat or return it through a result PR before closing the runtime. If the browser blocks or interrupts the download, run this final cell again; it reuses the verified archive.

If the runtime dies after packaging, the same ZIP is on Drive at `MyDrive/baomai-reviewed-annotations-seed46-20261111/baomai-reviewed-annotations-seed46-results.zip`.


In [ ]:
assert RESULTS_ZIP.is_file(), "Finish the packaging cell first."
from google.colab import files
files.download(str(RESULTS_ZIP))


### What happens after the return

After both arms finish, run fresh epoch173/control/candidate inference under one compatible clean evaluation revision. Use the same immutable implementation for this seed's inference, quality diagnostics and audit. Report original and PR30 revised labels separately at **box confidence 0.25 primary**, with **0.15 and 0.40 diagnostic**. Endpoint confidence remains0.50 and matching IoU 0.50.

Missed objects remain the first priority. Report false positives, missing visible endpoints, both-corner success at **5 and 10 original pixels**, hidden-side outputs and common-returned-endpoint errors. Both-corner denominators include all eligible reference objects, including misses. The 10 px radius remains provisional.

Use `scripts/rescore_annotation_comparison.py --recipe reviewed-annotations-seed46-v1` for this pair. Keep seed 45's report and pinned replay source separate; do not pool the repeated development images as new independent evidence or select whichever seed is favorable. No further seed search is part of this confirmation, and neither training completion nor two seed results establish independent-scene accuracy.
